# Comparison of Placement Algorithms

This notebook compares the performance metrics (latency and network usage) between two YAFS placement algorithms:
1. **Latency-based (Baseline)**
2. **Custom Proposed Strategy**

The analysis uses simulation trace data generated by YAFS.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Define file paths
results_dir = 'resultados'

# Latency Strategy Files
latency_trace_file = os.path.join(results_dir, '4-latency-sim_trace.csv')
latency_link_file = os.path.join(results_dir, '4-latency-sim_trace_link.csv')

# Custom Strategy Files
custom_trace_file = os.path.join(results_dir, '4-custom_proposed_by_felipe-sim_trace.csv')
custom_link_file = os.path.join(results_dir, '4-custom_proposed_by_felipe-sim_trace_link.csv')

# Load DataFrames
df_latency_trace = pd.read_csv(latency_trace_file)
df_latency_link = pd.read_csv(latency_link_file)

df_custom_trace = pd.read_csv(custom_trace_file)
df_custom_link = pd.read_csv(custom_link_file)

print("Data loaded successfully.")
print(f"Latency Trace: {df_latency_trace.shape}")
print(f"Custom Trace: {df_custom_trace.shape}")

In [ ]:
# Add Strategy Label
df_latency_trace['Strategy'] = 'Latency (Baseline)'
df_custom_trace['Strategy'] = 'Custom Proposed'

df_latency_link['Strategy'] = 'Latency (Baseline)'
df_custom_link['Strategy'] = 'Custom Proposed'

# Combine DataFrames
df_trace_all = pd.concat([df_latency_trace, df_custom_trace])
df_link_all = pd.concat([df_latency_link, df_custom_link])

# Filter for End-to-End Latency (Service calling Actuator)
# This message usually signifies the completion of a request in this topology
df_results = df_trace_all[df_trace_all['message'] == 'Service calling Actuator'].copy()

# Calculate Latency
df_results['latency'] = df_results['time_reception'] - df_results['time_emit']

print("Combined Data:")
print(df_results[['Strategy', 'latency']].head())

In [ ]:
# Visualize Latency Distribution
plt.figure(figsize=(10, 6))
sns.boxplot(x='Strategy', y='latency', data=df_results, palette="Set2")
plt.title('End-to-End Latency Distribution by Strategy')
plt.ylabel('Latency (time units)')
plt.xlabel('Placement Strategy')
plt.show()

# Violin Plot for density
plt.figure(figsize=(10, 6))
sns.violinplot(x='Strategy', y='latency', data=df_results, palette="Set2")
plt.title('End-to-End Latency Density by Strategy')
plt.ylabel('Latency (time units)')
plt.xlabel('Placement Strategy')
plt.show()

In [ ]:
# Calculate Total Network Usage (Bytes Transferred)
network_usage = df_link_all.groupby('Strategy')['size'].sum().reset_index()
network_usage.columns = ['Strategy', 'Total Bytes Transferred']

print(network_usage)

# Visualize Network Usage
plt.figure(figsize=(8, 5))
sns.barplot(x='Strategy', y='Total Bytes Transferred', data=network_usage, palette="viridis")
plt.title('Total Network Usage by Strategy')
plt.ylabel('Total Bytes Transferred')
plt.xlabel('Placement Strategy')
plt.show()

In [ ]:
# Latency Over Time
plt.figure(figsize=(12, 6))
sns.lineplot(x='time_reception', y='latency', hue='Strategy', data=df_results, alpha=0.7)
plt.title('Latency Over Time')
plt.xlabel('Simulation Time')
plt.ylabel('Latency')
plt.show()

In [ ]:
# Summary Statistics
summary_stats = df_results.groupby('Strategy')['latency'].agg(['mean', 'min', 'max', 'std', 'count']).reset_index()
summary_stats = summary_stats.merge(network_usage, on='Strategy')

print("Summary Statistics Table:")
display(summary_stats)